### **Student Information**
GitHub ID:

Kaggle name:

Kaggle private scoreboard snapshot: 

![pic_ranking.png](./pics/pic_ranking.png)

*Your name and student ID are deliberately not asked for here: this notebook goes into your repository, which is public. They belong in the git-ignored `config/.env`, where the other two notebooks read them from, and your NTU COOL submission is what ties the repository link back to you.*

---

# **Instructions**

This lab has **three parts**: exercises in the Master notebook, exercises and an agent conversation in the AgentPipeline notebook, and an internal Kaggle competition documented in this Homework notebook. Together they add up to **100 points**.

**Environment recommendations to solve lab 2:**
- **Part 1 and 2 exercises:** Everything can be run with only CPU.
- **Part 3 exercises:** For the competition you will probably need GPU to train your models, so it is recommended to use Colab or Kaggle if you don't have a laptop with a dedicated GPU.

## **Part 1 (40 pts):**

1. __Main Exercises (35 pts):__ Do the **take home exercises** from Sections: `1. Data Preparation` to `8. Information Retrieval and RAG`, in the DM2026-Lab2-Master Notebook. Total: `13 exercises`. Commit your code and submit the repository link to NTU Cool.

2. **Code Comments (5 pts):** **Tidy up the code in your notebook**. 

## **Part 2 (30 pts):**

1. **Agent Prompt (20 pts):** Interact with the Agent using appropriate prompts to guide it through and complete all the required tasks.

2. **Main Exercises (10 pts):** Do the remaining **take home exercises** from DM2026-Lab2-AgentPipeline Notebook. Total: `3 exercises required`. Commit your code and submit the repository link to NTU Cool.

## **Part 3 (30 pts):**

1. **Kaggle Competition Participation (15 pts):** Submit your predictions to the [Data Mining Class Kaggle Competition](https://www.kaggle.com/t/d99def2e62194b0d8f633267c32e6d1e) by **November 22**. Check Kaggle for the exact closing time. The competition concerns emotion recognition on Twitter; participation points use the Private Leaderboard ranking:
    - If your submission meets or exceeds the **baseline score (F1 score: 0.5)**, and there are no    significant issues with your report or code, you will automatically receive a base score of 5 points of the 15 points.

        The remaining 10 points will be determined by your ranking among participants who meet or exceed the baseline:

        |Ranking  |Additional Points  |
        | --- | --- |
        |Top 10% |+10 |
        |Top 20% |+9 |
        |Top 30% |+8 |
        |Top 40% |+7 |
        |Top 50% |+6 |
        |Top 60% |+5 |
        |Top 70% |+4 |
        |Top 80% |+3 |
        |Top 90% |+2 |
        |<Top 90% |+1 |

    - If your submission does not reach the baseline score, your grade will be lower than 5. The score will range from 0 to 4 points, depending on your Private Leaderboard ranking and the relative performance of students whose submissions are also below the baseline.

2. **Competition Report (15 pts)** A report section to be filled in inside this notebook in Markdown Format, we already provided you with the template below. You need to describe your work developing the model for the competition. The report should include a section describing briefly the following elements: 
* All of your AI prompts and what you decide do use.
* Your preprocessing steps.
* The feature engineering steps.
* Explanation of your model.

[Markdown Guide - Basic Syntax](https://www.markdownguide.org/basic-syntax/)

**`Things to note for Part 3:`**

* **The code used for the competition should be in this Jupyter Notebook File** `DM2026-Lab2-Homework.ipynb`.

* **Push the code used for the competition to your repository**.

* **The code should have a clear separation for the same sections of the report, AI prompts, preprocessing, feature engineering and model explanation. Briefly comment your code for easier understanding, we provide a template at the end of this notebook.**

* Save a screenshot of your Kaggle Private Leaderboard ranking as `pics/pic_ranking.png` and keep the image link in the Student Information cell at the top of this notebook.

The Kaggle competition closes on **November 22**. The three days afterward are for completing this notebook's Markdown report, checking your files, and submitting the lab by **November 25 at 11:59 pm (Wednesday)**.

Push your work to your repository, then submit its link and a ZIP on the corresponding NTU COOL assignment. The ZIP must contain `DM2026-Lab2-Master.ipynb`, `DM2026-Lab2-AgentPipeline.ipynb`, `DM2026-Lab2-Homework.ipynb`, both encrypted logs in `session_logs/`, and `pics/pic_ranking.png`. Name it with your student ID and your name. The two logs are git-ignored, so they are not in your repository and the ZIP is the only copy that reaches us: open it before uploading and check both are inside. If you miss them, you can upload them on NTU COOL until **November 26 at 11:59 pm**. See `README.md` for the exact file paths and setup steps.

---

Next you will find the template report with some simple markdown syntax explanations, use it to structure your content.

You can delete the syntax suggestions after you use them.

---

***

# **Project Report**

---

## 1. Model Development (10 pts Required)

The task is to classify short Reddit-style posts into six emotions: joy, anger, sadness, disgust, surprise and fear. The labeled training set has 3,360 posts and the test set has 1,440. The classes are imbalanced (joy 875 posts, fear only 288), and most posts are short (median of 13 words), so there is little text to work with per post. All code for this report is in the sections below, and every number quoted here comes from running the notebook with a fixed seed (42).

### 1.1 Preprocessing Steps

* **Merging the files.** The posts (`final_posts.json`), the labels (`emotion.csv`) and the train/test flags (`data_identification.csv`) were joined on `post_id`. The test posts have no labels, and a check in the notebook confirms that no labeled row ended up in the test set.
* **Light text cleaning.** Posts were lowercased, URLs were removed, and extra whitespace was squeezed. Letters or punctuation repeated three or more times were shortened to two (`sooooo` becomes `soo`, `!!!!!` becomes `!!`), so that stretched spellings share a vocabulary entry with the normal word while still keeping some of the emphasis. The anonymised placeholders `[NAME]` and `[RELIGION]` were turned into plain tokens, because the square brackets would otherwise be thrown away by the tokenizer.
* **What was deliberately kept.** Stop words, `!`, `?` and emoji were not removed. In emotion text these carry signal ("what?!" reads as surprise), and the posts are too short to lose words for free. No stemming was used either.
* **Duplicate texts.** 16 texts appear in the training set with more than one different label. Their label is unreliable, so all copies were dropped. Texts that were repeated with the same label were kept once. This took the training set from 3,360 to 3,294 posts. Test posts were never dropped, since every one needs a prediction.
* **Hold-out split.** A stratified 80/20 split of the 3,294 posts gives 2,635 posts for model selection and 659 posts that are only used once at the end, to get an honest score. The final model is refit on all 3,294 posts before predicting the test set.

### 1.2 Feature Engineering Steps

Three blocks of features are placed side by side in one sparse matrix:

* **Word TF-IDF (unigrams and bigrams).** Each word gets a weight that is high when the word is frequent in a post but rare across posts. Bigrams such as `not happy` capture some word order, which single words miss. `sublinear_tf=True` stops a word repeated many times from dominating, and `min_df=2` ignores words that appear in only one post.
* **Character TF-IDF (2 to 5 characters, inside word boundaries).** Small pieces of words let the model work with misspellings, emoji and unusual names that never appeared as whole words in training. These two TF-IDF blocks give 29,561 columns in total.
* **MiniLM sentence embeddings (384 columns).** A pre-trained encoder (`all-MiniLM-L6-v2`) turns each post into 384 numbers that reflect its meaning, so `terrified` and `scared` end up close together even though TF-IDF sees them as unrelated columns. The encoder is frozen, so it learns nothing from the labels and can be run on every post at once without leaking information. It was applied to the original text, not the cleaned text, because it was trained on natural text. The embeddings are multiplied by a weight (`emb_weight`) so that the dense block and the TF-IDF blocks have a comparable influence, and this weight is tuned in the grid search.

The TF-IDF vocabularies are learned inside a scikit-learn `Pipeline`, so in each cross-validation fold they are fitted on the training part only. This avoids leaking vocabulary or document-frequency information from the validation fold.

### 1.3 Explanation of Your Model

The final model is a **linear SVM (`LinearSVC`) with `class_weight="balanced"`** on the combined features, with `C = 0.1` and an embedding weight of 1.0. A linear model is a good match here, because the feature matrix has around 30,000 columns but only about 2,600 training posts. In this setting a linear model trains in seconds and is less prone to overfitting, while a more flexible model could more easily memorise the training posts. `class_weight="balanced"` raises the penalty for mistakes on small classes such as fear, so that the model does not simply favour joy and anger. A small `C` means stronger regularisation, and the search picked `C = 0.1` for both LinearSVC models.

Four candidates were compared with 5-fold stratified cross-validation on the 2,635 selection posts, scored with macro F1 (the average of the six per-class F1 scores, so small classes count as much as large ones):

| Model | CV macro F1 |
| --- | --- |
| TF-IDF + MiniLM + LinearSVC | **0.547** |
| TF-IDF + MiniLM + Logistic Regression | 0.536 |
| MiniLM only + Logistic Regression | 0.486 |
| TF-IDF only + LinearSVC | 0.478 |

The two single-family rows work as an ablation. Neither feature family is enough alone, but combining them raised macro F1 by about 0.07 over TF-IDF alone. A plausible reason is that the two blocks make different kinds of errors: TF-IDF matches the exact words and spellings, and MiniLM matches meaning even when the words differ. This is an interpretation of the scores and was not tested further. The linear SVM was slightly better than logistic regression on the same features.

On the 659 held-out posts, which were not used for any choice, the selected model scored **macro F1 0.572, weighted F1 0.596 and accuracy 0.598**. Per class, joy was the easiest (F1 0.72), followed by surprise (0.63) and sadness (0.60). Fear (0.48) and disgust (0.46) were the hardest, which fits their smaller size and the fact that their wording overlaps with anger and sadness. The Kaggle leaderboard metric is not stated in the materials, so macro F1 was optimised and weighted F1 is reported next to it. The final model was refit on all 3,294 labeled posts to produce `submission.csv`.

Limitations: the validation set is small (56 fear posts, for example), so the per-class scores are noisy. Only one frozen encoder was tried, and no fine-tuned transformer was tested.

**Kaggle private leaderboard result:** [add private F1 and ranking after the competition closes]

### 1.4 AI Prompts, what and why you decide to use that technique that suggest by AI

OpenCode was used as an AI coding assistant during the competition to explore possible approaches, obtain technical suggestions, and support model development. The following describes its main contribution to the development process.

**Prompt 1:**

*"Based on the competition requirements, dataset, and provided reference materials, suggest suitable approaches for data preprocessing, feature engineering, and model implementation. Explain the possible techniques and their advantages so that they can be evaluated for this task."*

The AI assistant reviewed the competition requirements, existing notebooks, README, and provided datasets. It identified differences between the competition dataset and the SemEval dataset used in the Master notebook, particularly the number of classes (six instead of four), data format, and class imbalance.

It suggested three possible approaches:

1. **TF-IDF with a linear classifier.**
2. **Transformer fine-tuning.**
3. **An ensemble combining both approaches.**

**Decision:** TF-IDF with a linear classifier was selected as the initial approach because it had shown promising results in the reference notebook and was computationally efficient for experimentation and cross-validation. Transformer fine-tuning and ensembling were initially postponed due to computational limitations, as the installed PyTorch environment only supported CPU execution.

During subsequent experimentation, several techniques were explored and evaluated:

- **Macro F1-score:** Selected as the primary evaluation metric because it gives equal importance to each class, including minority classes. Weighted F1-score was also retained for comparison.

- **MiniLM embeddings:** Introduced as an additional feature representation alongside TF-IDF. Cross-validation results improved from 0.478 to 0.547, supporting its inclusion in the final model.

- **Character n-grams:** Added to capture word patterns and variations that might not be fully represented by word-level features.

- **Class weighting:** `class_weight="balanced"` was applied to help address the imbalanced class distribution.

- **Pipeline and cross-validation:** A Pipeline was used to keep preprocessing within the cross-validation process and reduce data leakage risks. A separate hold-out set was also maintained for final evaluation.

During data inspection, duplicated texts with conflicting labels were identified. Instead of arbitrarily selecting one label, all copies of the 16 conflicting duplicated texts were excluded to avoid introducing uncertain labels into training.

Overall, AI suggestions were treated as possible approaches to investigate rather than automatically accepted solutions. The techniques included in the final implementation were selected based on their relevance to the dataset, computational feasibility, and experimental performance.

During data inspection, duplicated texts with conflicting labels were identified. Instead of arbitrarily selecting one label, all copies of the 16 conflicting duplicated texts were excluded to avoid introducing uncertain labels into training.

Overall, AI suggestions were treated as possible approaches to investigate rather than automatically accepted solutions. The techniques included in the final implementation were selected based on their relevance to the dataset, computational feasibility, and experimental performance.

**`From here on starts the code section for the competition.`**

---

# **Competition Code**

## 1. Preprocessing Steps

In [ ]:
import json
import os
import re
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy.sparse as sp

warnings.filterwarnings("ignore")

# One seed for every random step (CV folds, hold-out split, classifiers).
SEED = 42
np.random.seed(SEED)

# The competition ships three files: the posts, the labels (train only), and
# which post belongs to the train or test set. Merge them on post_id.
DATA_DIR = "kaggle_data"
posts = pd.DataFrame(json.load(open(f"{DATA_DIR}/final_posts.json", encoding="utf8")))
labels = pd.read_csv(f"{DATA_DIR}/emotion.csv")
split = pd.read_csv(f"{DATA_DIR}/data_identification.csv")
sample_sub = pd.read_csv(f"{DATA_DIR}/sampleSubmission.csv")

df = posts.merge(split, on="post_id").merge(labels, on="post_id", how="left")
print("all posts:", df.shape, "| split sizes:", df["set"].value_counts().to_dict())
print("train rows without a label:", df.loc[df["set"] == "train", "label"].isna().sum())
print("test rows with a label (should be 0):", df.loc[df["set"] == "test", "label"].notna().sum())
print(df.loc[df["set"] == "train", "label"].value_counts().to_dict())

In [ ]:
# Light cleaning only. Emotion lives in punctuation ("!!!", "?!"), emoji and
# word choice, so we keep those and only remove noise that splits the vocabulary.
def clean_text(s: str) -> str:
    s = s.lower()
    s = re.sub(r"\[name\]", " xname ", s)           # anonymised name placeholder -> plain token
    s = re.sub(r"\[religion\]", " xreligion ", s)   # same for the religion placeholder
    s = re.sub(r"http\S+|www\.\S+", " ", s)         # URLs
    s = re.sub(r"(.)\1{2,}", r"\1\1", s)            # "sooooo" -> "soo", "!!!!!" -> "!!"
    return re.sub(r"\s+", " ", s).strip()            # squeeze whitespace

df["clean"] = df["text"].map(clean_text)

for i in [11, 15]:
    print("RAW:  ", df["text"].iloc[i])
    print("CLEAN:", df["clean"].iloc[i], "\n")

In [ ]:
train_all = df[df["set"] == "train"].copy()
test_df = df[df["set"] == "test"].reset_index(drop=True)   # test rows are never dropped: every one needs a prediction

# Duplicate training texts: if the same text carries two different labels the
# label is unreliable, so all copies go. Identical text with the same label is
# kept once, so a repeated post is not counted twice in CV and in the model.
n_before = len(train_all)
label_counts = train_all.groupby("clean")["label"].nunique()
conflicting = label_counts[label_counts > 1].index
train_all = train_all[~train_all["clean"].isin(conflicting)].drop_duplicates("clean")
train_all = train_all.reset_index(drop=True)
print(f"conflicting-label texts removed: {len(conflicting)}")
print(f"training rows: {n_before} -> {len(train_all)}")

# Stratified 80/20 hold-out. Model selection (grid search, CV) uses only the 80%
# part; the 20% part is scored once at the end as an honest estimate. After that
# the chosen model is refit on all labeled data for the Kaggle submission.
from sklearn.model_selection import train_test_split
fit_df, val_df = train_test_split(train_all, test_size=0.2, stratify=train_all["label"], random_state=SEED)
fit_df, val_df = fit_df.reset_index(drop=True), val_df.reset_index(drop=True)
print("fit:", fit_df.shape, "| validation:", val_df.shape, "| test:", test_df.shape)

## 2. Feature Engineering Steps

In [ ]:
# MiniLM sentence embeddings (384 numbers per post). The encoder is frozen and
# pre-trained, so nothing is fitted on our labels and encoding all posts at once
# leaks nothing. The raw text is encoded, not the cleaned text, because the
# encoder was trained on natural text. Cached to disk: encoding takes seconds
# on a CPU, but there is no reason to repeat it.
from sentence_transformers import SentenceTransformer

def encode_cached(texts, cache_path):
    if os.path.exists(cache_path):
        cached = np.load(cache_path)
        if len(cached) == len(texts):
            return cached
    model = SentenceTransformer("all-MiniLM-L6-v2")
    vecs = model.encode(list(texts), batch_size=64, show_progress_bar=False, normalize_embeddings=True)
    os.makedirs(os.path.dirname(cache_path), exist_ok=True)
    np.save(cache_path, vecs)
    return vecs

# Delete data/embeddings/kaggle_*.npy if the rows or the encoder ever change.
EMB_COLS = [f"e{i}" for i in range(384)]

def with_embeddings(frame, name):
    emb = encode_cached(frame["text"], f"data/embeddings/kaggle_minilm_{name}.npy")
    return pd.concat([frame[["clean"]].reset_index(drop=True), pd.DataFrame(emb, columns=EMB_COLS)], axis=1)

X_fit, y_fit = with_embeddings(fit_df, "fit"), fit_df["label"].to_numpy()
X_val, y_val = with_embeddings(val_df, "val"), val_df["label"].to_numpy()
X_all, y_all = with_embeddings(train_all, "all"), train_all["label"].to_numpy()
X_test = with_embeddings(test_df, "test")
print("feature frames:", X_fit.shape, X_val.shape, X_all.shape, X_test.shape)

In [ ]:
# The full feature builder: three blocks stacked side by side into one sparse matrix.
#   word TF-IDF  : unigrams + bigrams ("not happy"), sublinear tf so repeated words do not dominate
#   char TF-IDF  : 2-5 character pieces inside word edges; handles typos, emoji, odd names
#   MiniLM       : 384 dense meaning features, multiplied by emb_weight to balance them against TF-IDF
# It lives inside the Pipeline below, so the TF-IDF vocabularies are learned
# from the training part of each CV fold only (no leakage).
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import FeatureUnion

def build_features(emb_weight=1.0, use_tfidf=True, use_emb=True):
    blocks, weights = [], {}
    if use_tfidf:
        tfidf = FeatureUnion([
            ("word", TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True)),
            ("char", TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), min_df=2, sublinear_tf=True)),
        ])
        blocks.append(("tfidf", tfidf, "clean"))
    if use_emb:
        blocks.append(("emb", "passthrough", EMB_COLS))
        weights["emb"] = emb_weight
    # sparse_threshold=1.0 keeps the result sparse even with the 384 dense columns
    return ColumnTransformer(blocks, transformer_weights=weights, sparse_threshold=1.0)

demo = build_features().fit(X_fit)
M = demo.transform(X_fit)
print("combined feature matrix (fit part):", M.shape, "| sparse:", sp.issparse(M))
print("word + char TF-IDF columns:", M.shape[1] - len(EMB_COLS), "| MiniLM columns:", len(EMB_COLS))

## 3. Model Implementation Steps

In [ ]:
# Model comparison, 5-fold stratified CV on the 80% fit part, scored with macro
# F1 (the mean of the six per-emotion F1 scores, so small classes such as fear
# count as much as big ones). class_weight="balanced" counters the class imbalance.
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

def make_pipe(clf, **feat_kwargs):
    return Pipeline([("features", build_features(**feat_kwargs)), ("clf", clf)])

svc = lambda: LinearSVC(class_weight="balanced", random_state=SEED)
lr = lambda: LogisticRegression(class_weight="balanced", max_iter=3000, random_state=SEED)

# Each entry: pipeline + the hyper-parameters to search. The first two are ablations
# (what does each feature family give on its own?), the last two are the real candidates.
candidates = {
    "TF-IDF only + LinearSVC":    (make_pipe(svc(), use_emb=False), {"clf__C": [0.03, 0.1, 0.3, 1]}),
    "MiniLM only + LogReg":       (make_pipe(lr(), use_tfidf=False), {"clf__C": [1, 3, 10]}),
    "TF-IDF + MiniLM + LinearSVC": (make_pipe(svc()), {"clf__C": [0.03, 0.1, 0.3],
                                                      "features__transformer_weights": [{"emb": 0.5}, {"emb": 1.0}, {"emb": 1.5}]}),
    "TF-IDF + MiniLM + LogReg":   (make_pipe(lr()), {"clf__C": [1, 3, 10],
                                                    "features__transformer_weights": [{"emb": 1.0}, {"emb": 2.0}]}),
}

searches, rows = {}, []
for name, (pipe, grid) in candidates.items():
    t0 = time.time()
    gs = GridSearchCV(pipe, grid, scoring="f1_macro", cv=cv, n_jobs=1, refit=True).fit(X_fit, y_fit)
    searches[name] = gs
    rows.append({"model": name, "best CV macro F1": round(gs.best_score_, 4),
                 "best params": gs.best_params_, "seconds": round(time.time() - t0)})

cv_results = pd.DataFrame(rows).sort_values("best CV macro F1", ascending=False).reset_index(drop=True)
print(cv_results.to_string(index=False))

In [ ]:
# Score the CV winner once on the 20% hold-out it never saw. Accuracy and
# weighted F1 are shown too, since the leaderboard F1 variant is not stated.
from sklearn.metrics import (ConfusionMatrixDisplay, accuracy_score, classification_report,
                             confusion_matrix, f1_score)

best_name = cv_results.loc[0, "model"]
best_search = searches[best_name]
y_val_pred = best_search.predict(X_val)

print("Best candidate by CV:", best_name, best_search.best_params_)
print(f"hold-out accuracy   : {accuracy_score(y_val, y_val_pred):.4f}")
print(f"hold-out macro F1   : {f1_score(y_val, y_val_pred, average='macro'):.4f}")
print(f"hold-out weighted F1: {f1_score(y_val, y_val_pred, average='weighted'):.4f}\n")
print(classification_report(y_val, y_val_pred))

CLASSES = sorted(set(y_all))
fig, ax = plt.subplots(figsize=(5.5, 5))
ConfusionMatrixDisplay(confusion_matrix(y_val, y_val_pred, labels=CLASSES), display_labels=CLASSES).plot(
    ax=ax, cmap="Blues", colorbar=False, xticks_rotation=45)
ax.set_title(f"Hold-out confusion matrix\n{best_name}")
plt.tight_layout(); plt.show()

In [ ]:
import torch
from torch.utils.data import DataLoader
from transformers import AutoModelForSequenceClassification, AutoTokenizer, get_linear_schedule_with_warmup

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if DEVICE == "cpu":
    print("WARNING: no GPU found, fine-tuning will be very slow. Turn on a GPU accelerator.")

PRETRAINED = "j-hartmann/emotion-english-distilroberta-base"
EPOCHS, LR, BATCH, MAX_LEN, WARMUP = 4, 3e-5, 16, 128, 0.1
TR_SEEDS = [42, 43, 44]
CLASSES = sorted(set(y_all))
cls2id = {c: i for i, c in enumerate(CLASSES)}
tok = AutoTokenizer.from_pretrained(PRETRAINED)

def make_loader(texts, labels=None, shuffle=False):
    enc = tok(list(texts), truncation=True, max_length=MAX_LEN)
    items = []
    for i in range(len(enc["input_ids"])):
        item = {"input_ids": enc["input_ids"][i], "attention_mask": enc["attention_mask"][i]}
        if labels is not None:
            item["labels"] = cls2id[labels[i]]
        items.append(item)
    return DataLoader(items, batch_size=BATCH if shuffle else 64, shuffle=shuffle,
                      collate_fn=lambda b: tok.pad(b, return_tensors="pt"))

def load_model():
    model = AutoModelForSequenceClassification.from_pretrained(PRETRAINED)
    src = {k.lower(): v for k, v in model.config.label2id.items()}
    if all(c in src for c in CLASSES):
        # keep the pre-trained weights of the six emotions we need, drop the rest
        old = model.classifier.out_proj
        new = torch.nn.Linear(old.in_features, len(CLASSES))
        idx = [src[c] for c in CLASSES]
        new.weight.data = old.weight.data[idx].clone()
        new.bias.data = old.bias.data[idx].clone()
        model.classifier.out_proj = new
        model.config.num_labels = len(CLASSES)
        return model
    return AutoModelForSequenceClassification.from_pretrained(
        PRETRAINED, num_labels=len(CLASSES), ignore_mismatched_sizes=True)

def finetune_predict(train_texts, train_labels, eval_texts, seed):
    """Fine-tune on the training posts, return class probabilities for eval_texts."""
    torch.manual_seed(seed); np.random.seed(seed)
    model = load_model().to(DEVICE)
    loader = make_loader(train_texts, list(train_labels), shuffle=True)

    counts = pd.Series(train_labels).value_counts()
    weights = torch.tensor([len(train_labels) / (len(CLASSES) * counts[c]) for c in CLASSES],
                           dtype=torch.float, device=DEVICE)
    loss_fn = torch.nn.CrossEntropyLoss(weight=weights)

    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
    total = EPOCHS * len(loader)
    sched = get_linear_schedule_with_warmup(opt, int(WARMUP * total), total)

    model.train()
    for _ in range(EPOCHS):
        for batch in loader:
            batch = {k: v.to(DEVICE) for k, v in batch.items()}
            labels = batch.pop("labels")
            loss = loss_fn(model(**batch).logits, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step(); sched.step(); opt.zero_grad()

    model.eval()
    out = []
    with torch.no_grad():
        for batch in make_loader(eval_texts):
            batch = {k: v.to(DEVICE) for k, v in batch.items()}
            out.append(torch.softmax(model(**batch).logits, dim=-1).cpu().numpy())
    del model, opt
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
    return np.vstack(out)

# Train on the 80% fit part only, score on the 20% hold-out.
t0 = time.time()
val_probs_tr = np.mean([finetune_predict(fit_df["text"], y_fit, val_df["text"], s) for s in TR_SEEDS], axis=0)
print(f"fine-tuning ({len(TR_SEEDS)} seeds) took {time.time() - t0:.0f}s")

# Classical model for the blend: LogReg gives probabilities (LinearSVC does not).
LR_NAME = "TF-IDF + MiniLM + LogReg"
val_probs_lr = searches[LR_NAME].best_estimator_.predict_proba(X_val)
assert list(searches[LR_NAME].best_estimator_.classes_) == CLASSES

# Blend weight = share of the transformer. Only 3 values are tried, so there is little room to overfit.
blend_scores = {}
for w in [0.5, 0.7, 1.0]:
    pred = np.array(CLASSES)[(w * val_probs_tr + (1 - w) * val_probs_lr).argmax(1)]
    blend_scores[w] = f1_score(y_val, pred, average="macro")
    print(f"transformer weight {w}: hold-out macro F1 = {blend_scores[w]:.4f}")
print(f"(TF-IDF + MiniLM classical best, for reference: {f1_score(y_val, y_val_pred, average='macro'):.4f})")

BLEND_W = max(blend_scores, key=blend_scores.get)
y_val_blend = np.array(CLASSES)[(BLEND_W * val_probs_tr + (1 - BLEND_W) * val_probs_lr).argmax(1)]
print("\nchosen transformer weight:", BLEND_W)
print(f"hold-out accuracy: {accuracy_score(y_val, y_val_blend):.4f} | macro F1: {f1_score(y_val, y_val_blend, average='macro'):.4f}\n")
print(classification_report(y_val, y_val_blend))

In [ ]:
# Final models: same settings as on the hold-out, refit on ALL labeled posts
# (fit + hold-out), then predict the Kaggle test posts with the chosen blend.
from sklearn.base import clone

lr_final = clone(searches[LR_NAME].best_estimator_).fit(X_all, y_all)
test_probs_lr = lr_final.predict_proba(X_test)
test_probs_tr = np.mean([finetune_predict(train_all["text"], y_all, test_df["text"], s) for s in TR_SEEDS], axis=0)

test_probs = BLEND_W * test_probs_tr + (1 - BLEND_W) * test_probs_lr
test_pred = np.array(CLASSES)[test_probs.argmax(1)]

# Kaggle's scorer looks for a column named "id" (the sample file calls it post_id),
# so the submission uses "id" for the ID and keeps "label" for the prediction.
ID_COL, LABEL_COL = "id", "label"
print("submission columns:", [ID_COL, LABEL_COL])

pred = pd.DataFrame({ID_COL: test_df["post_id"], LABEL_COL: test_pred})
# Keep exactly the row order of the sample submission file.
submission = sample_sub[["post_id"]].rename(columns={"post_id": ID_COL}).merge(pred, on=ID_COL, how="left")

assert list(submission.columns) == [ID_COL, LABEL_COL]
assert len(submission) == len(sample_sub) and submission[ID_COL].equals(sample_sub["post_id"])
assert submission[LABEL_COL].notna().all() and set(submission[LABEL_COL]) <= set(CLASSES)

submission.to_csv("submission.csv", index=False)

print("saved submission.csv:", submission.shape)
print(submission[LABEL_COL].value_counts(normalize=True).round(3).to_dict())
print("train label share  :", pd.Series(y_all).value_counts(normalize=True).round(3).to_dict())
submission.head()